# **grib2io: Kerchunk and Icechunk Integration**

This notebook demonstrates how to use `grib2io` to create virtual Zarr datasets from GRIB2 files using **Kerchunk** and **Icechunk**. These technologies allow for cloud-optimized, lazy access to archival GRIB2 data without duplicating the underlying data.

**NOTE**: This notebook uses the specialized `grib2io.kerchunk` and `grib2io.icechunk` modules, which are designed to work seamlessly with the `grib2io` xarray backend.

In [ ]:
import xarray as xr
import grib2io
from grib2io.kerchunk import ReferenceGenerator
from grib2io.icechunk import IcechunkWriter
import os
import json

## **1. Generating Kerchunk Reference Manifests**

Kerchunk creates a reference manifest (JSON or Parquet) that maps Zarr chunks to byte ranges within the original GRIB2 files. `grib2io.kerchunk.ReferenceGenerator` leverages `grib2io`'s indexing infrastructure to create these manifests efficiently.

In [ ]:
# Path to a sample GRIB2 file from the test suite
input_data_dir = "../tests/input_data"
grib_file = os.path.join(input_data_dir, "gfs.jpeg.grib2")

# Initialize the ReferenceGenerator
gen = ReferenceGenerator(grib_file)

# Generate the manifest (in-memory dict)
manifest = gen.generate()

# Save the manifest to a JSON file
json_manifest = "gfs_manifest.json"
gen.to_json(json_manifest)
print(f"Manifest saved to {json_manifest}")

## **2. Opening Kerchunk Manifests with Xarray**

The `grib2io` xarray backend provides integrated support for Kerchunk. By specifying `engine="grib2io" `, the backend automatically handles the `fsspec` reference filesystem setup and uses the specialized `Grib2Codec` to decode chunks on-the-fly.

In [ ]:
# Open the dataset via the Kerchunk manifest using the grib2io engine
ds_kerchunk = xr.open_dataset(json_manifest, engine="grib2io")
display(ds_kerchunk)

## **3. Using Icechunk for Versioned Virtual Stores**

**Icechunk** is a transactional storage engine for Zarr. `grib2io.icechunk.IcechunkWriter` translates Kerchunk manifests into Icechunk's Zarr v3-based virtual store format. This allows for version control and efficient data management.

In [ ]:
# Define a path for the local Icechunk store
icechunk_path = "gfs_icechunk_store"

# Initialize the IcechunkWriter
writer = IcechunkWriter(icechunk_path)

# Write the manifest to the store
writer.write(manifest)

# Commit the changes to create a versioned snapshot
snapshot_id = writer.commit("Initial ingest of GFS data")
print(f"Committed snapshot: {snapshot_id}")

## **4. Opening Icechunk Stores with Xarray**

Icechunk stores can also be opened directly with `xr.open_dataset` using the `grib2io` engine. The backend detects the Icechunk repository and manages the session internally.

In [ ]:
# Open the dataset from the Icechunk store
ds_icechunk = xr.open_dataset(icechunk_path, engine="grib2io")
display(ds_icechunk)

## **5. Appending Data to Icechunk**

Icechunk makes it easy to extend datasets. You can append new GRIB2 files to an existing virtual store along a specific dimension (e.g., time or lead time).

In [ ]:
# Path to another file to append
append_grib_file = os.path.join(input_data_dir, "gfs.complex.grib2")
new_gen = ReferenceGenerator(append_grib_file)
new_manifest = new_gen.generate()

# Write to existing store in append mode
# Note: we specify the dimension along which to append, e.g., 'refDate'
writer = IcechunkWriter(icechunk_path)
writer.write(new_manifest, mode="a", append_dim="refDate")
writer.commit("Appended new GRIB2 file along refDate")

# Re-open to see the extended dataset
ds_extended = xr.open_dataset(icechunk_path, engine="grib2io")
print(f"Extended dataset refDate size: {ds_extended.refDate.size}")